# 🟢 KaizenStat — Basic Demo (5 min)

**Level:** Beginner | **Time:** ~5 minutes | **Dataset:** Titanic

This notebook teaches the **core 3-step flow**:
1. Load data → `fit()`
2. Check health → `health()`
3. Train a model → `train()`

No prior ML experience needed.

---
> **What you'll learn:**
> - How to get a Data Health Score in one line
> - How KaizenStat auto-selects the best model for you
> - How to read the training output

In [ ]:
!pip install kaizenstat -q
print("✅ Ready")

## 1. Load the Titanic Dataset

The Titanic dataset predicts whether a passenger **survived** (1) or **died** (0).

We drop `PassengerId`, `Name`, `Ticket`, and `Cabin` — these are ID/free-text columns
with no predictive value that would confuse the ML pipeline.

In [ ]:
import pandas as pd
from kaizenstat import DataDoctor

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Drop ID / free-text columns — no predictive value
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

print(f"Shape: {df.shape[0]} passengers, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

## 2. Register with DataDoctor

`fit()` auto-detects task type (classification) and dataset mode (tabular).

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target="Survived")

print(f"\nMode: {doctor.mode()}")

## 3. Data Health Score

Score is 0–100. Interpretation:
- **90–100**: Excellent
- **70–89**: Minor issues — fix before production
- **50–69**: Significant issues — fix before training
- **< 50**: Major problems — training will likely give poor results

In [ ]:
health = doctor.health()
print(f"\n📊 Health Score: {health.score} / 100")

## 4. Fix Data Issues

`fix(safe=True)` fills missing `Age` with median, `Embarked` with mode, and removes any duplicates.

In [ ]:
fixed_df = doctor.fix(safe=True)

print(f"Before fix: {df.isnull().sum().sum()} missing values")
print(f"After fix:  {fixed_df.isnull().sum().sum()} missing values")

## 5. Train

`train()` benchmarks multiple ML algorithms, picks the best one, and trains it with cross-validation.
You don't need to choose an algorithm — KaizenStat picks it for you.

In [ ]:
train_result = doctor.train(cv=5)

print(f"\n🏆 Best model:  {train_result.model_name}")
print(f"   Test score:  {train_result.test_score:.4f}")
print(f"   Train score: {train_result.train_score:.4f}")

gap = train_result.train_score - train_result.test_score
if gap > 0.1:
    print(f"\n⚠️  Overfitting (gap={gap:.3f}) — try more data or regularisation")
else:
    print(f"\n✅ Healthy train/test gap ({gap:.3f})")

## 6. Generate Report

In [ ]:
report_path = doctor.report(output_path="basic_report.html")
print(f"📄 Report saved: {report_path}")

from IPython.display import IFrame, display
display(IFrame(src='basic_report.html', width='100%', height='500px'))

## 🎯 Try It Yourself

**Exercise 1:** Predict passenger class instead
```python
doctor2 = DataDoctor()
doctor2.fit(df, target="Pclass")
doctor2.train()
```

**Exercise 2:** Try with your own CSV
```python
my_df = pd.read_csv("your_file.csv")
doctor3 = DataDoctor()
doctor3.fit(my_df, target="your_target")
doctor3.health()
doctor3.train()
```

**Exercise 3:** Enable hyperparameter tuning
```python
tuned = doctor.train(tune=True, n_iter=20)
print(f"Tuned score: {tuned.test_score:.4f}")
```

---
## What's Next?

| Notebook | Level | What you'll learn |
|----------|-------|-------------------|
| **You are here** | 🟢 Basic | fit → health → train → report |
| [Intermediate (15 min)](demo_intermediate.ipynb) | 🟡 Intermediate | Full 8-step + debug + trust score |
| [Advanced (30 min)](demo_advanced.ipynb) | 🔴 Advanced | Tuning + custom models + feature impact + codegen |
| [Quick Start](quickstart_tabular.ipynb) | ⚡ Reference | All 8 steps in one clean notebook |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*